# Advanced Financial QA System with RAG

## Hybrid Retrieval-Augmented Generation for Financial Question Answering

This notebook implements a **Financial QA system** that addresses three fundamental challenges:

### Core Capabilities
1. **Automated Numerical Reasoning** - Program-of-Thought execution for verifiable computations
2. **Implicit Temporal Reasoning** - Temporal graph construction for time-aware QA
3. **Causality Detection** - Causal relation extraction from financial narratives
4. **Hybrid Retrieval** - Dual-index (dense + BM25) retrieval for tables and text

### Architecture
- **LLM Backend**: Llama 3.2 (open-source, quantized for efficiency)
- **Retrieval**: Sentence-Transformers + FAISS + BM25 hybrid
- **Reasoning**: Program-of-Thought numerical execution, temporal graphs, causal pattern matching
- **Evaluation**: Comprehensive metrics for each reasoning capability

### Dataset
- **FinQA**: Financial question answering over structured tables and unstructured text from financial reports


## 1. Installation

In [ ]:
%%capture
# Install all required dependencies
!pip install transformers>=4.35.0
!pip install torch>=2.0.0
!pip install sentence-transformers>=2.2.0
!pip install faiss-cpu
!pip install datasets pandas numpy scikit-learn
!pip install networkx>=3.0
!pip install nltk>=3.8.0
!pip install accelerate>=0.24.0
!pip install bitsandbytes>=0.41.0
!pip install tqdm
!pip install huggingface-hub>=0.19.0

## 2. Imports and Configuration

In [ ]:
import os
import sys
import json
import re
import math
import time
import random
import warnings
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Ensure src is importable
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, '.')

# Check available hardware
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch version: {torch.__version__}")

## 3. Download and Load FinQA Dataset

In [ ]:
from src.data.finqa_loader import (
    download_finqa_dataset,
    load_finqa_dataset,
    load_finqa_split,
    FinQAExample,
    classify_question_type,
)

# Download and load dataset
dataset = load_finqa_dataset(
    data_dir="./finqa_data",
    download=True,
    max_train=500,   # Limit for faster demo
    max_dev=100,
    max_test=100,
)

print(f"\nDataset loaded:")
for split, examples in dataset.items():
    print(f"  {split}: {len(examples)} examples")

## 4. Explore Dataset Structure

In [ ]:
# Examine a sample example
if 'train' in dataset and dataset['train']:
    ex = dataset['train'][0]
    print(f"ID: {ex.id}")
    print(f"Question: {ex.question}")
    print(f"Answer: {ex.answer}")
    print(f"Program: {ex.program_str}")
    print(f"\nTable ({len(ex.table)} rows):")
    for row in ex.table[:5]:
        print(f"  {row}")
    print(f"\nPre-text (first 2):")
    for t in ex.pre_text[:2]:
        print(f"  {t[:100]}...")
    print(f"\nPost-text (first 2):")
    for t in ex.post_text[:2]:
        print(f"  {t[:100]}...")

In [ ]:
# Analyze question type distribution
from src.data.finqa_loader import classify_question_type

type_counts = defaultdict(int)
multi_type = 0

train_examples = dataset.get('train', [])
for ex in train_examples:
    qtypes = classify_question_type(ex.question, ex.program)
    active = [k for k, v in qtypes.items() if v]
    if len(active) > 1:
        multi_type += 1
    for t in active:
        type_counts[t] += 1

print("Question Type Distribution:")
for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {t}: {c} ({c/len(train_examples)*100:.1f}%)")
print(f"  multi-type: {multi_type} ({multi_type/len(train_examples)*100:.1f}%)")

## 5. Initialize System Components

### 5.1 Hybrid Retrieval System
Dense (Sentence-Transformers + FAISS) + Sparse (BM25) retrieval for both tables and text.


In [ ]:
from src.retrieval.hybrid_retriever import HybridRetriever, FinancialDocumentIndexer

# Initialize retriever
retriever = HybridRetriever(
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    bm25_weight=0.3,
    dense_weight=0.7,
)

# Index training examples for few-shot retrieval
indexer = FinancialDocumentIndexer(retriever)
num_indexed = indexer.index_examples(train_examples[:200])
print(f"Indexed {num_indexed} document chunks from {min(200, len(train_examples))} examples")

### 5.2 Numerical Reasoning Module (Program-of-Thought)
Generates and executes programs instead of relying on LLM arithmetic.


In [ ]:
from src.reasoning.numerical_reasoner import NumericalReasoner

numerical_reasoner = NumericalReasoner(
    execution_timeout=5,
    tolerance=1e-4,
)

# Demo: Parse and execute a FinQA DSL program
demo_program = ["subtract(5829, 5735)", "divide(#0, 5735)", "multiply(#1, 100)"]
steps = numerical_reasoner.parse_finqa_program(demo_program)
result = numerical_reasoner.execute_program(steps)

print("Program-of-Thought Demo:")
print(f"  Program: {demo_program}")
print(f"  Steps:")
for s in result['steps']:
    print(f"    Step {s['step']}: {s['raw']} -> args={s['args']} -> result={s['result']}")
print(f"  Final Answer: {result['result']}")
print(f"  Success: {result['success']}")

### 5.3 Temporal Reasoning Module
Extracts temporal entities, builds temporal graphs, detects trends.


In [ ]:
from src.reasoning.temporal_reasoner import TemporalReasoner

temporal_reasoner = TemporalReasoner(max_hops=3)

# Demo: Temporal analysis
demo_question = "What was the percentage increase in revenue from 2019 to 2021?"
demo_table = [
    ["Item", "2018", "2019", "2020", "2021"],
    ["Revenue", "$50,000", "$60,000", "$75,000", "$90,000"],
    ["Net Income", "$10,000", "$12,000", "$15,000", "$18,000"],
]

temp_result = temporal_reasoner.reason(
    question=demo_question,
    table=demo_table,
    context="The company showed consistent growth in revenue over the period.",
)

print("Temporal Reasoning Demo:")
print(f"  Question: {demo_question}")
print(f"  Entities: {temp_result['temporal_entities']}")
print(f"  Temporal types: {temp_result['temporal_type']}")
print(f"  Ordering: {temp_result['temporal_ordering']}")
print(f"  Context: {temp_result['temporal_context']}")
if temp_result['trend_analysis']:
    print(f"  Trend: {temp_result['trend_analysis']['trend']}")

### 5.4 Causality Detection Module
Extracts cause-effect relationships from financial narratives.


In [ ]:
from src.reasoning.causality_detector import CausalityDetector

causality_detector = CausalityDetector(
    confidence_threshold=0.4,
    max_causal_hops=2,
)

# Demo: Causal extraction
demo_text = (
    "Operating costs increased due to supply chain disruptions in Q3. "
    "The higher costs led to a decline in operating margin. "
    "Revenue growth was driven by market expansion in Asia-Pacific."
)

causal_result = causality_detector.reason(
    question="Why did operating costs increase?",
    context=demo_text,
)

print("Causality Detection Demo:")
print(f"  Is causal question: {causal_result['is_causal']}")
print(f"  Relations found: {len(causal_result['causal_relations'])}")
for rel in causal_result['causal_relations']:
    print(f"    {rel['cause']} -> {rel['effect']} "
          f"(conf={rel['confidence']:.2f}, type={rel['relation_type']})")
print(f"\n  Causal context:\n{causal_result['causal_context']}")

## 6. Question Classifier

Routes questions to appropriate reasoning modules based on pattern analysis.


In [ ]:
from src.reasoning.question_classifier import QuestionClassifier

classifier = QuestionClassifier()

# Demo classification
demo_questions = [
    "What was the percentage growth in operating income from 2021 to 2023?",
    "Why did the company's profit decline in 2022?",
    "Which quarter showed the highest revenue growth?",
    "What is the total revenue for 2021?",
]

print("Question Classification Demo:")
print("-" * 70)
for q in demo_questions:
    scores = classifier.classify(q)
    primary = classifier.get_primary_type(q)
    active = classifier.get_active_modules(q)
    print(f"Q: {q}")
    print(f"  Primary: {primary}")
    print(f"  Active modules: {active}")
    print(f"  Scores: { {k: f'{v:.2f}' for k, v in scores.items()} }")
    print()

## 7. Integrated Pipeline

The full Financial QA pipeline combining all modules.


In [ ]:
from src.pipeline import FinancialQAPipeline

# Initialize the full pipeline
# Set load_llm=True and provide a valid model name to use Llama
pipeline = FinancialQAPipeline(
    model_name="meta-llama/Llama-3.2-1B-Instruct",  # Change to your preferred model
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    load_llm=False,  # Set True if you have GPU and model access
    load_in_4bit=True,
)

print("Pipeline initialized successfully")
print(f"  LLM available: {pipeline.llm.is_available}")
print(f"  Modules: classifier, retriever, numerical_reasoner, temporal_reasoner, causality_detector")

## 8. Run Pipeline on Examples

### 8.1 Single Example Demo


In [ ]:
# Run on first few examples
dev_examples = dataset.get('dev', dataset.get('train', []))[:5]

print("Running pipeline on sample examples...\n")
sample_results = []

for ex in dev_examples:
    result = pipeline.answer(ex)
    sample_results.append(result)

    print("=" * 70)
    print(f"ID: {result['id']}")
    print(f"Question: {result['question']}")
    print(f"Gold Answer: {result['gold_answer']}")
    print(f"Predicted: {result['predicted_answer']}")
    print(f"Primary Type: {result['classification']['primary_type']}")
    print(f"Active Modules: {result['classification']['active_modules']}")
    print(f"Time: {result['time_seconds']:.3f}s")
    print(f"Reasoning trace:")
    for trace in result['reasoning_trace']:
        print(f"  - {trace}")
    print()

### 8.2 Custom Example

In [ ]:
# Test with a custom financial example
custom_example = FinQAExample(
    id="custom_1",
    question="What was the percentage increase in operating income from 2019 to 2020?",
    table=[
        ["Item", "2018", "2019", "2020", "2021"],
        ["Revenue", "$50,000", "$60,000", "$75,000", "$90,000"],
        ["Operating Income", "$8,000", "$10,000", "$13,500", "$16,200"],
        ["Net Income", "$5,000", "$6,500", "$9,000", "$11,000"],
        ["Total Assets", "$100,000", "$120,000", "$150,000", "$180,000"],
    ],
    pre_text=[
        "The company demonstrated strong operational performance across all segments.",
        "Operating income growth was primarily driven by margin improvements.",
    ],
    post_text=[
        "Management expects continued growth in the upcoming fiscal year.",
    ],
    program=["subtract(13500, 10000)", "divide(#0, 10000)", "multiply(#1, 100)"],
    answer="35.0",
)

result = pipeline.answer(custom_example)

print("Custom Example Results:")
print("=" * 70)
print(f"Question: {result['question']}")
print(f"Gold Answer: {result['gold_answer']}")
print(f"Predicted: {result['predicted_answer']}")
print(f"Classification: {result['classification']}")
print(f"\nNumerical reasoning:")
num = result.get('numerical', {})
if num.get('program_steps'):
    for step in num['program_steps']:
        print(f"  Step {step['step']}: {step['raw']} = {step['result']}")
print(f"\nTemporal reasoning:")
temp = result.get('temporal', {})
print(f"  Entities: {temp.get('temporal_entities', [])}")
print(f"  Context: {temp.get('temporal_context', '')}")
print(f"\nCausal reasoning:")
causal = result.get('causal', {})
print(f"  Is causal: {causal.get('is_causal', False)}")
print(f"  Relations: {len(causal.get('causal_relations', []))}")

## 9. Batch Evaluation with Comprehensive Metrics

Evaluates the system across four key dimensions:
1. **Numerical Reasoning** - Execution accuracy, relative error
2. **Context Filtering** - Retrieval precision, recall, F1, sufficiency
3. **Causality Detection** - Detection rate, confidence scores
4. **Temporal Reasoning** - Temporal score, trend detection rate


In [ ]:
from src.evaluation.metrics import FinQAEvaluator

evaluator = FinQAEvaluator(tolerance=0.01)

# Evaluate on dev set (or subset)
eval_examples = dataset.get('dev', dataset.get('train', []))[:50]

print(f"Evaluating on {len(eval_examples)} examples...")
start_time = time.time()

eval_results = pipeline.batch_answer(eval_examples, verbose=True)

elapsed = time.time() - start_time
print(f"Evaluation completed in {elapsed:.1f}s ({elapsed/len(eval_examples):.2f}s per example)")

In [ ]:
# Generate comprehensive evaluation report
report = evaluator.evaluate(eval_results, eval_examples)

# Print the report
evaluator.print_report(report)

## 10. Detailed Metrics Analysis

### 10.1 Numerical Reasoning Metrics


In [ ]:
# Detailed numerical reasoning analysis
print("=" * 70)
print("NUMERICAL REASONING - DETAILED ANALYSIS")
print("=" * 70)

numerical_examples = [r for r in eval_results if 'numerical' in r.get('classification', {}).get('active_modules', [])]
print(f"\nQuestions requiring numerical reasoning: {len(numerical_examples)}")

# Analyze execution accuracy by program complexity
complexity_buckets = defaultdict(list)
for r in numerical_examples:
    num_info = r.get('numerical', {})
    n_steps = len(num_info.get('program_steps', []))
    bucket = f"{n_steps}-step" if n_steps <= 3 else "4+-step"
    match = 1.0 if r.get('predicted_answer') and r.get('gold_answer') and \
        evaluator.numerical_metrics.execution_accuracy(
            r['predicted_answer'], r['gold_answer']
        )['exact_match'] else 0.0
    complexity_buckets[bucket].append(match)

print("\nAccuracy by program complexity:")
for bucket in sorted(complexity_buckets.keys()):
    scores = complexity_buckets[bucket]
    print(f"  {bucket}: {np.mean(scores):.4f} ({sum(int(s) for s in scores)}/{len(scores)})")

# Analyze by operation type
op_accuracy = defaultdict(list)
for r in numerical_examples:
    num_info = r.get('numerical', {})
    for step in num_info.get('program_steps', []):
        op = step.get('operation', 'unknown')
        op_accuracy[op].append(1.0 if r.get('predicted_answer') == r.get('gold_answer') else 0.0)

print("\nAccuracy by operation type:")
for op in sorted(op_accuracy.keys()):
    scores = op_accuracy[op]
    print(f"  {op}: {np.mean(scores):.4f} (n={len(scores)})")

### 10.2 Context Filtering Metrics

In [ ]:
print("=" * 70)
print("CONTEXT FILTERING - DETAILED ANALYSIS")
print("=" * 70)

ctx_metrics = report['context_filtering']
print(f"\nRetrieval Performance:")
print(f"  Mean Precision: {ctx_metrics['mean_precision']:.4f}")
print(f"  Mean Recall:    {ctx_metrics['mean_recall']:.4f}")
print(f"  Mean F1:        {ctx_metrics['mean_f1']:.4f}")
print(f"  Mean Sufficiency: {ctx_metrics['mean_sufficiency']:.4f}")

# Analyze retrieval quality vs answer accuracy
high_suff = []
low_suff = []
for r in eval_results:
    retrieval = r.get('retrieval', {})
    all_ctx = []
    for ctx in retrieval.get('table_contexts', []):
        all_ctx.append(ctx.get('text', ''))
    for ctx in retrieval.get('text_contexts', []):
        all_ctx.append(ctx.get('text', ''))

    if all_ctx:
        full_ctx = " ".join(all_ctx)
        suff = evaluator.context_metrics.context_sufficiency(
            full_ctx, r.get('question', ''), r.get('gold_answer', '')
        )
        is_correct = r.get('predicted_answer') and r.get('gold_answer') and \
            evaluator.numerical_metrics.execution_accuracy(
                r['predicted_answer'], r['gold_answer']
            )['exact_match']

        if suff['sufficiency_score'] >= 0.5:
            high_suff.append(float(is_correct))
        else:
            low_suff.append(float(is_correct))

print(f"\nAnswer Accuracy by Retrieval Quality:")
print(f"  High sufficiency (>=0.5): {np.mean(high_suff):.4f} (n={len(high_suff)})" if high_suff else "  High sufficiency: N/A")
print(f"  Low sufficiency (<0.5):   {np.mean(low_suff):.4f} (n={len(low_suff)})" if low_suff else "  Low sufficiency: N/A")

### 10.3 Causality Detection Metrics

In [ ]:
print("=" * 70)
print("CAUSALITY DETECTION - DETAILED ANALYSIS")
print("=" * 70)

causal_metrics = report['causality_detection']
print(f"\nCausal Questions Found: {causal_metrics['causal_questions_found']}")
print(f"Detection Rate: {causal_metrics['detection_rate']:.4f}")
print(f"Mean Confidence: {causal_metrics['mean_confidence']:.4f}")
print(f"Total Relations: {causal_metrics['total_relations_detected']}")
print(f"Avg Relations/Question: {causal_metrics['avg_relations_per_question']:.2f}")

# Show sample causal detections
causal_examples = [r for r in eval_results if r.get('causal', {}).get('is_causal')]
print(f"\nSample Causal Detections (up to 3):")
for r in causal_examples[:3]:
    print(f"  Q: {r['question'][:80]}...")
    for rel in r['causal'].get('causal_relations', [])[:2]:
        print(f"    {rel['cause'][:40]} -> {rel['effect'][:40]} (conf={rel['confidence']:.2f})")
    print()

### 10.4 Temporal Reasoning Metrics

In [ ]:
print("=" * 70)
print("TEMPORAL REASONING - DETAILED ANALYSIS")
print("=" * 70)

temp_metrics = report['temporal_reasoning']
print(f"\nTemporal Questions Found: {temp_metrics['temporal_questions_found']}")
print(f"Mean Temporal Score: {temp_metrics['mean_temporal_score']:.4f}")
print(f"Mean Entities Extracted: {temp_metrics['mean_entities_extracted']:.2f}")
print(f"Trend Detection Rate: {temp_metrics['trend_detection_rate']:.4f}")

# Show sample temporal reasoning
temporal_examples = [r for r in eval_results if 'temporal' in r.get('classification', {}).get('active_modules', [])]
print(f"\nSample Temporal Reasoning (up to 3):")
for r in temporal_examples[:3]:
    temp = r.get('temporal', {})
    print(f"  Q: {r['question'][:80]}...")
    print(f"    Entities: {temp.get('temporal_entities', [])[:3]}")
    print(f"    Types: { {k:v for k,v in temp.get('temporal_type', {}).items() if v} }")
    print(f"    Context: {temp.get('temporal_context', 'N/A')[:80]}")
    print()

## 11. Performance Summary Dashboard


In [ ]:
# Create a summary dashboard
print("\n" + "=" * 70)
print("       FINANCIAL QA SYSTEM - PERFORMANCE DASHBOARD")
print("=" * 70)

overall_acc = report['overall']['accuracy']
num_acc = report['numerical_reasoning'].get('execution_accuracy', 0)
ctx_f1 = report['context_filtering'].get('mean_f1', 0)
causal_det = report['causality_detection'].get('detection_rate', 0)
temp_score = report['temporal_reasoning'].get('mean_temporal_score', 0)

print(f"""
+-------------------------------+----------+
| Metric                        | Score    |
+-------------------------------+----------+
| Overall QA Accuracy           | {overall_acc:>6.4f}   |
| Numerical Execution Accuracy  | {num_acc:>6.4f}   |
| Context Retrieval F1          | {ctx_f1:>6.4f}   |
| Causality Detection Rate      | {causal_det:>6.4f}   |
| Temporal Reasoning Score      | {temp_score:>6.4f}   |
+-------------------------------+----------+

Per-Question-Type Breakdown:
""")

for qtype, info in report.get('per_type_accuracy', {}).items():
    bar = '#' * int(info['accuracy'] * 30)
    print(f"  {qtype:<12} [{bar:<30}] {info['accuracy']:.4f} ({info['correct']}/{info['count']})")

print(f"""
Total examples evaluated: {report['num_examples']}
""")

## 12. Architecture Summary

### System Components

```
                    +-------------------+
                    | Question Input    |
                    +--------+----------+
                             |
                    +--------v----------+
                    | Question Classifier|
                    | (pattern-based)   |
                    +--------+----------+
                             |
              +--------------+--------------+
              |              |              |
     +--------v---+  +------v------+  +---v----------+
     | Numerical  |  | Temporal    |  | Causality    |
     | Reasoner   |  | Reasoner   |  | Detector     |
     | (PoT exec) |  | (graph)    |  | (pattern+NN) |
     +--------+---+  +------+------+  +---+----------+
              |              |              |
              +--------------+--------------+
                             |
                    +--------v----------+
                    | Hybrid Retriever  |
                    | (Dense + BM25)    |
                    +--------+----------+
                             |
                    +--------v----------+
                    | Evidence          |
                    | Aggregation       |
                    +--------+----------+
                             |
                    +--------v----------+
                    | LLM Answer Gen    |
                    | (Llama 3.2)       |
                    +--------+----------+
                             |
                    +--------v----------+
                    | Final Answer      |
                    +-------------------+
```

### Key Design Decisions

1. **Program-of-Thought over implicit arithmetic**: Generates executable Python code for numerical reasoning, eliminating LLM arithmetic hallucinations.

2. **Temporal graphs over flat retrieval**: Constructs temporal ordering graphs to reason about financial event sequences, trends, and period comparisons.

3. **Pattern + domain knowledge for causality**: Combines linguistic patterns with financial domain knowledge for causal relation extraction.

4. **Hybrid retrieval**: Dense semantic search (FAISS) + sparse lexical search (BM25) ensures both semantic and keyword-level matching for financial documents.


In [ ]:
print("Notebook execution complete.")
print("To enable LLM-powered answers, set load_llm=True in the pipeline initialization.")
print("Recommended model: meta-llama/Llama-3.2-1B-Instruct (or 3B/8B for better quality)")